# Dimensional Coverage Analysis (Experiment 2)

Analyze how well emergent themes cover the 10 strategic dimensions.

## Prerequisites

1. Run the backfill script to add dimensional coverage to existing traces:
   ```bash
   npx tsx scripts/backfill-dimensional-coverage.ts
   ```

2. Make sure you have your virtual environment activated and dependencies installed:
   ```bash
   python3 -m venv venv
   source venv/bin/activate
   pip install -r requirements.txt
   ```

3. Have `DATABASE_URL` in your `.env.local`

In [ ]:
# Setup: Load dimensional coverage functions
import sys
sys.path.append('..')  # Add parent directory to path

from scripts.dimensional_coverage_analysis import (
    load_dimensional_coverage,
    analyze_coverage_patterns,
    find_systematic_gaps,
    get_coverage_summary_stats,
    get_theme_to_dimension_mapping
)

import pandas as pd

# Load all traces with dimensional coverage
df_coverage = load_dimensional_coverage()
print(f"Loaded {len(df_coverage)} traces with dimensional coverage\n")

if len(df_coverage) == 0:
    print("⚠️  No traces with dimensional coverage found!")
    print("\nRun the backfill script first:")
    print("  npx tsx scripts/backfill-dimensional-coverage.ts")

In [ ]:
# Overall Summary Statistics
if len(df_coverage) > 0:
    summary = get_coverage_summary_stats(df_coverage)
    
    print("=" * 60)
    print("OVERALL DIMENSIONAL COVERAGE SUMMARY")
    print("=" * 60)
    print(f"Total traces analyzed: {summary['total_traces']}")
    print(f"Avg dimensions covered: {summary['avg_dimensions_covered']:.1f} / 10")
    print(f"Avg coverage percentage: {summary['avg_coverage_percentage']:.1f}%")
    print(f"Avg primary (high confidence) dimensions: {summary['avg_primary_dimensions']:.1f}")
    print("=" * 60)

In [ ]:
# Coverage Patterns by Dimension
if len(df_coverage) > 0:
    coverage_patterns = analyze_coverage_patterns(df_coverage)
    
    print("\n" + "=" * 60)
    print("DIMENSION COVERAGE PATTERNS")
    print("=" * 60 + "\n")
    print(coverage_patterns.to_string(index=False))
    
    # Optional: Visualize with matplotlib
    try:
        import matplotlib.pyplot as plt
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        # Coverage rate by dimension
        ax1.barh(coverage_patterns['dimension'], coverage_patterns['coverage_rate'])
        ax1.set_xlabel('Coverage Rate')
        ax1.set_title('Dimension Coverage Rate')
        ax1.set_xlim(0, 1)
        ax1.grid(axis='x', alpha=0.3)
        
        # Average confidence when covered
        covered_dims = coverage_patterns[coverage_patterns['coverage_rate'] > 0]
        if len(covered_dims) > 0:
            ax2.barh(covered_dims['dimension'], covered_dims['avg_confidence_when_covered'])
            ax2.set_xlabel('Avg Confidence (1=low, 2=medium, 3=high)')
            ax2.set_title('Average Confidence When Covered')
            ax2.set_xlim(0, 3)
            ax2.grid(axis='x', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    except ImportError:
        print("\n(Install matplotlib for visualizations: pip install matplotlib)")

In [ ]:
# Systematic Gaps (dimensions covered in <50% of conversations)
if len(df_coverage) > 0:
    gaps = find_systematic_gaps(df_coverage, threshold=0.5)
    
    print("\n" + "=" * 60)
    print("SYSTEMATIC GAPS (Coverage < 50%)")
    print("=" * 60 + "\n")
    
    if len(gaps) > 0:
        print(f"Found {len(gaps)} dimensions that are systematically under-covered:\n")
        print(gaps.to_string(index=False))
        
        print("\n⚠️  These dimensions may need proactive questioning in future experiments")
        print("   Consider: Experiment 3 - Proactive Gap-Based Questioning")
    else:
        print("✅ No systematic gaps! All dimensions covered in >50% of conversations")

In [ ]:
# Individual Trace Detail (example)
if len(df_coverage) > 0:
    # Pick first trace as example (change index to explore different traces)
    trace_idx = 0  # Change this to look at different traces
    trace_row = df_coverage.iloc[trace_idx]
    coverage = trace_row['dimensionalCoverage']
    
    print("\n" + "=" * 60)
    print(f"TRACE DETAIL (#{trace_idx + 1} of {len(df_coverage)})")
    print("=" * 60)
    print(f"Trace ID: {trace_row['trace_id']}")
    print(f"Timestamp: {trace_row['timestamp']}")
    print(f"Variant: {trace_row['experimentVariant']}")
    print(f"\nCoverage: {coverage['summary']['dimensionsCovered']}/10 dimensions ({coverage['summary']['coveragePercentage']}%)")
    print(f"Primary (high confidence): {len(coverage['summary']['primaryDimensions'])}")
    print(f"Gaps: {len(coverage['summary']['gaps'])}")
    
    print("\n" + "=" * 60)
    print("DIMENSION BREAKDOWN")
    print("=" * 60 + "\n")
    
    for dimension, data in coverage['dimensions'].items():
        status = "✅" if data['covered'] else "❌"
        conf = data['confidence'].upper() if data['covered'] else "N/A"
        themes = ", ".join(data['themes']) if data['themes'] else "None"
        
        print(f"{status} {dimension}")
        if data['covered']:
            print(f"   Confidence: {conf}")
            print(f"   Themes: {themes}")
        print()
    
    if coverage['summary']['gaps']:
        print(f"\n⚠️  Missing dimensions: {', '.join(coverage['summary']['gaps'])}")

In [ ]:
# Theme to Dimension Mapping
if len(df_coverage) > 0:
    theme_mappings = get_theme_to_dimension_mapping(df_coverage)
    
    print("\n" + "=" * 60)
    print("THEME → DIMENSION MAPPINGS")
    print("=" * 60 + "\n")
    print(f"Total mappings: {len(theme_mappings)}\n")
    
    if len(theme_mappings) > 0:
        # Show sample mappings
        print("Sample mappings (first 20):\n")
        print(theme_mappings.head(20).to_string(index=False))
        
        # Most common dimension mappings
        print("\n\nMost commonly mapped dimensions:")
        dimension_counts = theme_mappings['dimension'].value_counts()
        for dim, count in dimension_counts.head(10).items():
            print(f"  {dim}: {count} theme mappings")